# Sector Rotation on HOSE

Pull HOSE sector returns, rank them quarterly on trailing 3-month performance, and hold the top 3 equal-weighted. Benchmark against VN-Index.

**Why this matters**: sector rotation is the simplest non-trivial allocation strategy that actually works in Vietnam. Cross-sectional momentum at the sector level is far less noisy than at the single-name level, and turnover stays low.

## Setup

Data is mocked for offline execution. Replace each mock with the corresponding `dc.dataset(...)` call when you have credentials.

In [ ]:
import pandas as pd  # noqa
import numpy as np  # noqa
import matplotlib.pyplot as plt

rng = np.random.default_rng(2025)
plt.rcParams['figure.figsize'] = (11, 6)

## Step 1 — Pull sector total-return series

Replace with `dc.dataset('equity.hose.sector_returns').to_pandas(start='2018-01-01')`. We'll mock 7 years of monthly returns for 10 HOSE sectors with realistic drift and dispersion.

In [ ]:
sectors = ['Banks', 'Real Estate', 'Consumer Staples', 'Consumer Disc.',
           'Energy', 'Materials', 'Technology', 'Utilities',
           'Industrials', 'Healthcare']

dates = pd.date_range('2018-01-31', '2025-04-30', freq='ME')

# Each sector: drift ~ 0.4-1.4%/mo, vol ~ 5-9%/mo
drifts = rng.uniform(0.004, 0.014, len(sectors))
vols   = rng.uniform(0.05, 0.09, len(sectors))

sector_rets = pd.DataFrame(
    rng.normal(loc=drifts, scale=vols, size=(len(dates), len(sectors))),
    index=dates, columns=sectors,
)

# VN-Index as cap-weighted-ish blend
weights_vni = np.array([0.32, 0.22, 0.10, 0.06, 0.08, 0.07, 0.05, 0.04, 0.04, 0.02])
vni = (sector_rets * weights_vni).sum(axis=1).rename('VN-Index')

print(sector_rets.shape, 'monthly sector returns')
sector_rets.tail(3).round(3)

## Step 2 — Quarterly ranking on trailing 3-month return

At each quarter-end we look back over the last 3 months, rank sectors, pick the top 3, and hold them through the next quarter.

In [ ]:
# Trailing 3-month compounded return at each month-end
trail_3m = (1 + sector_rets).rolling(3).apply(np.prod, raw=True) - 1

# Resample to quarter-end signal
signal = trail_3m.resample('QE').last()

# Top 3 each quarter
top3 = signal.apply(lambda row: row.nlargest(3).index.tolist(), axis=1)
top3.tail(6)

## Step 3 — Build the strategy returns

For each month, look up the most recent quarter-end's top-3 list and equal-weight those sectors' returns.

In [ ]:
# Forward-fill quarterly picks onto monthly index
picks_monthly = top3.reindex(sector_rets.index, method='ffill')

def pf_return(row, picks):
    held = picks.loc[row.name]
    if not isinstance(held, list):
        return np.nan
    return row[held].mean()

strat = sector_rets.apply(lambda r: pf_return(r, picks_monthly), axis=1).rename('Top3 Rotation')
strat = strat.dropna()
strat.head()

## Step 4 — Equity curve vs VN-Index

In [ ]:
curves = pd.concat([
    (1 + strat).cumprod().rename('Top3 Rotation'),
    (1 + vni.loc[strat.index]).cumprod().rename('VN-Index'),
], axis=1)

fig, ax = plt.subplots()
curves.plot(ax=ax, lw=2)
ax.set_title('Sector rotation: top-3 trailing-3M, equal weight')
ax.set_ylabel('Growth of 1 VND')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Step 5 — Performance summary

Annualized return, vol, Sharpe, max drawdown for both lines.

In [ ]:
def stats(r):
    ann_ret = (1 + r).prod() ** (12 / len(r)) - 1
    ann_vol = r.std() * np.sqrt(12)
    sharpe  = ann_ret / ann_vol if ann_vol else np.nan
    eq = (1 + r).cumprod()
    mdd = (eq / eq.cummax() - 1).min()
    return pd.Series({'Ann Ret': ann_ret, 'Ann Vol': ann_vol,
                      'Sharpe': sharpe, 'Max DD': mdd})

summary = pd.concat({
    'Top3 Rotation': stats(strat),
    'VN-Index':      stats(vni.loc[strat.index]),
}, axis=1).T
summary.style.format({'Ann Ret': '{:.1%}', 'Ann Vol': '{:.1%}',
                       'Sharpe': '{:.2f}', 'Max DD': '{:.1%}'})

## Step 6 — Which sectors did the work?

Count how often each sector was selected. Concentrated picks tell you whether the alpha came from one persistent winner or a real rotation.

In [ ]:
picks_flat = pd.Series([s for picks in top3.dropna() for s in picks])
fig, ax = plt.subplots(figsize=(8, 4))
picks_flat.value_counts().plot(kind='barh', ax=ax, color='darkgreen')
ax.set_xlabel('# quarters held')
ax.set_title('Sector selection frequency')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Next steps

- `03-factor-investing/02-momentum-12-1.ipynb` — same idea, single names
- `01-vn30-fundamentals.ipynb` — within the chosen sectors, pick on quality
- `05-macro/02-fx-vnd-regime.ipynb` — condition rotation on FX regime